# 28 模型公共契约与资源保留

本教程演示：双 API 不泄漏目标列、字段顺序与概率语义、五档调参结果保留、按需读取、完整续跑制品与轻量推理制品。每一步都包含数值一致性断言。

依赖：`pip install -e ".[tune,dev]"`。优先使用本地 `hscredit_yyp.xlsx`；没有工作簿时使用固定种子的模拟数据。输出默认写入新建临时目录，不覆盖已有报告。


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import os
import sys

repository_root = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "hscredit").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("请从仓库根目录或 examples 目录启动 Notebook")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

import numpy as np
import pandas as pd
import optuna
from sklearn.base import clone
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from hscredit.core.models import RandomForest, ModelTuner, ProbabilityCalibrator, ProbabilityScoreCard

output_dir = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT") or mkdtemp(prefix="hscredit_model_workflow_"))
output_dir.mkdir(parents=True, exist_ok=True)
data_path = Path(os.environ.get("HSCREDIT_EXAMPLE_INPUT") or repository_root / "examples" / "hscredit_yyp.xlsx")
FEATURES = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
TARGET = "FPD"
optuna.logging.set_verbosity(optuna.logging.WARNING)


## 1. 准备训练和独立测试数据

缺失值填充参数仅由训练集计算。测试集不参与训练、搜索、校准或评分转换器拟合。


In [ ]:
if data_path.is_file():
    data = pd.read_excel(data_path)[FEATURES + [TARGET]].copy()
    source_name = "真实贷款数据"
else:
    values, labels = make_classification(n_samples=240, n_features=3, n_informative=3, n_redundant=0, random_state=42)
    data = pd.DataFrame(values, columns=FEATURES).assign(FPD=labels)
    source_name = "固定种子模拟数据"
data = data.replace([np.inf, -np.inf], np.nan).dropna(subset=[TARGET])
X_train, X_test, y_train, y_test = train_test_split(
    data[FEATURES], data[TARGET].astype(int), test_size=.2, stratify=data[TARGET], random_state=42,
)
medians = X_train.median().fillna(0)
X_train, X_test = X_train.fillna(medians), X_test.fillna(medians)
train_frame = X_train.assign(FPD=y_train)
display(pd.DataFrame({"数据来源": [source_name], "训练样本数": [len(X_train)], "测试样本数": [len(X_test)]}))


## 2. 双 API、字段顺序与评分契约

显式 y 优先，但 target 列始终从特征中移除。DataFrame 预测按训练字段顺序对齐。类别 1 表示坏样本，默认信用分越高风险越低。


In [ ]:
source_model = RandomForest(n_estimators=8, max_depth=3, random_state=42, n_jobs=1, target=TARGET)
implicit = clone(source_model).fit(train_frame)
explicit = clone(source_model).fit(train_frame.assign(FPD=1-y_train), y_train)
np.testing.assert_allclose(implicit.predict_proba(X_test), explicit.predict_proba(X_test.iloc[:, ::-1]))
assert TARGET not in explicit.feature_names_in_
probability = explicit.predict_proba(X_test)[:, 1]
scores = explicit.predict_score(X_test)
display(pd.DataFrame({"坏样本概率": probability, "信用评分": scores}).head())
display(pd.DataFrame([explicit.evaluate(X_test, y_test, metrics=["auc", "ks", "brier"], score_direction="higher_risk")]))


## 3. 校准和概率评分卡沿用相同字段契约

自动校准会克隆基础模型，重新划分训练集与校准集，不修改原始模型。一般 Pipeline 应使用 ProbabilityScoreCard 保留完整预处理；逐项 WOE 评分卡要求 LR 确实在 WOE 上训练。


In [ ]:
calibrator = ProbabilityCalibrator(model=source_model, target=TARGET, calib_ratio=.25, random_state=42)
calibrator.fit(train_frame, y_train)
np.testing.assert_allclose(calibrator.predict_proba(X_test), calibrator.predict_proba(X_test.iloc[:, ::-1]))
probability_card = ProbabilityScoreCard(model=calibrator, prefit=True, positive_class=1, pdo=50)
probability_card.fit(X_train)
np.testing.assert_allclose(
    probability_card.predict_score(X_test),
    probability_card.predict_score(proba=calibrator.predict_proba(X_test)[:, 1]),
)
display(calibrator.calibration_report(X_test, y_test))


## 4. 五档资源保留策略

| 策略 | 保留内容 |
|---|---|
| full | 全部折模型、预测、指标和训练记录；默认行为 |
| predictions | 保留预测、指标及精简训练记录，不保留折模型 |
| summary | 仅保留指标、曲线、异常、样本数等摘要，无 OOF 数组 |
| best | 仅当前最佳试验保留完整折结果，其余保留摘要 |
| disk | 每折单独写盘，内存仅保留索引；读取不会重新常驻折模型 |

未指定 retention 时，store_models=False 仍映射为 predictions。策略只改变结果保留，不改变搜索和训练。输入数据默认仍保留用于最终重训和续跑，后面演示显式释放。


In [ ]:
tuners = {}
comparison = []
reference_oof = None
reference_predictions = None
for mode in ["full", "predictions", "summary", "best", "disk"]:
    tuner = ModelTuner(
        RandomForest(n_estimators=5, random_state=42, n_jobs=1, target=TARGET),
        search_space={"max_depth": [2, 4]},
        trial_points=[{"max_depth": 2}, {"max_depth": 4}],
        objective="auc", objective_kwargs={"score_direction": "higher_risk"},
        target=TARGET, cv=2, n_jobs=1, random_state=42,
        retention=mode, artifact_dir=str(output_dir / mode),
    )
    tuner.fit(train_frame, n_trials=2, show_progress_bar=False)
    best = tuner.get_best_model()
    predicted = best.predict_proba(X_test)
    if reference_predictions is None:
        reference_predictions = predicted
    np.testing.assert_allclose(predicted, reference_predictions)
    if mode != "summary":
        oof = tuner.get_oof_predictions()["预测概率"].to_numpy()
        if reference_oof is None:
            reference_oof = oof
        np.testing.assert_allclose(oof, reference_oof)
    resident_models = sum("模型" in fold for result in tuner.trial_results_.values() for fold in result["各折"])
    comparison.append({"保留策略": mode, "最佳AUC": tuner.best_score_, "常驻折模型数": resident_models, "测试预测与full一致": True})
    tuners[mode] = tuner
display(pd.DataFrame(comparison))


## 5. 磁盘档按需读取与搜索过程分析

load=False 不读取折模型。显式请求详细结果时返回完整结果，但不放回常驻缓存。Optuna Study、各试验指标和可视化入口在所有模式下都保留。


In [ ]:
tuner = tuners["disk"]
number = tuner.best_trial_.number
index = tuner.get_trial_result(number, load=False)
display(pd.DataFrame([{key: fold.get(key) for key in ["折编号", "训练样本数", "验证样本数", "状态", "制品路径"]} for fold in index["各折"]]))
detail = tuner.get_trial_result(number)
assert all("模型" in fold for fold in detail["各折"])
assert all("模型" not in fold for fold in tuner.trial_results_[number]["各折"])
del detail
display(tuner.get_optimization_history())
# 可选交互图：tuner.visualization.plot_optimization_history().show()


## 6. 完整制品、轻量推理制品与保存往返

save 保留完整调参链；save_inference 移除 tuner 和已知训练记录，不修改原模型，也不改变预测与评分。Python 制品仅从可信来源加载，不保证任意依赖版本间迁移。disk 完整续跑还需保留 artifact_dir 中的折制品。


In [ ]:
best = tuner.get_best_model()
assert best.tuner is tuner
full_path = best.save(output_dir / "model_full.joblib")
inference_path = best.save_inference(output_dir / "model_inference.joblib")
restored = RandomForest.load(inference_path)
np.testing.assert_allclose(restored.predict_proba(X_test), best.predict_proba(X_test))
np.testing.assert_allclose(restored.predict_score(X_test), best.predict_score(X_test))
assert restored.tuner is None and best.tuner is tuner
display(pd.DataFrame({
    "制品": ["完整模型", "推理模型"],
    "文件字节数": [Path(full_path).stat().st_size, Path(inference_path).stat().st_size],
}))


## 7. 保存续跑，再显式释放训练输入

释放输入后仍可使用已缓存的最佳模型。需要重新拟合、组装完整标签的 OOF 或继续搜索时，重新提供输入和样本参数。释放不会删除折结果或磁盘制品。


In [ ]:
tuner_path = tuner.save(output_dir / "tuner.joblib")
resumed = ModelTuner.load(tuner_path)
resumed.fit(train_frame, n_trials=1, show_progress_bar=False)
assert len(resumed.study_.trials) == 3
final_model = resumed.get_best_model()
expected = final_model.predict_proba(X_test)
resumed.release_training_data()
assert resumed._X is None and resumed._y is None
np.testing.assert_allclose(resumed.get_best_model().predict_proba(X_test), expected)
try:
    resumed.get_best_model(refit=True)
except ValueError as error:
    assert "训练数据已释放" in str(error)
else:
    raise AssertionError("释放训练数据后不应默默重训")
print("公共契约、五档保留策略、保存恢复和续跑的语义检查全部通过")
